# Multi-Agent Research Persona Builder (Claude API)

Four agents work together to turn raw research data (interview notes, survey answers, observations) into evidence-based personas:

| Agent | Job |
|---|---|
| **Analyst** | Reads the raw data, extracts themes, goals, pain points, behaviours, with quotes as evidence |
| **Segmenter** | Groups participants into distinct user segments (one segment = one persona) |
| **Persona Writer** | Writes a full persona for each segment, grounded in the evidence |
| **Critic** | Reviews each persona for unsupported claims, stereotypes and overlap, then the Writer revises |

**How to run in Google Colab**
1. Click the 🔑 *Secrets* icon in the left sidebar, add a secret named `ANTHROPIC_API_KEY` with your key, and enable *Notebook access*.
2. Run the cells from top to bottom (Runtime → Run all).
3. Replace the sample data in Step 3 with your own (paste text or upload a `.txt` / `.csv`).

## Step 1 — Install the Anthropic SDK

In [ ]:
!pip install -q anthropic

## Step 2 — Connect to Claude

In [ ]:
import anthropic

try:
    from google.colab import userdata          # running in Colab
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ImportError:                             # running locally
    import os
    API_KEY = os.environ["ANTHROPIC_API_KEY"]

client = anthropic.Anthropic(api_key=API_KEY)
MODEL = "claude-opus-5-5"

def ask_agent(role_prompt: str, task: str, effort: str = "medium") -> str:
    """Send one task to one agent (a system prompt = the agent's role) and return its text answer."""
    with client.beta.messages.stream(
        model=MODEL,
        max_tokens=32000,
        system=role_prompt,
        messages=[{"role": "user", "content": task}],
        output_config={"effort": effort},           # low | medium | high | xhigh | max
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",                         # if a request is declined, retry on a fallback model
    ) as stream:
        response = stream.get_final_message()

    if response.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {response.stop_details}")
    return "".join(b.text for b in response.content if b.type == "text")

print(ask_agent("You are a helpful assistant.", "Reply with the single word: ready", effort="low"))

## Step 3 — Your research data

Paste your interview notes / survey responses below, **or** set `UPLOAD = True` to upload a file.
Also describe the product or research context so the agents know what the personas are for.

In [ ]:
RESEARCH_CONTEXT = """
We are designing a mobile app that helps university students manage their study time
and research projects.
"""

RAW_DATA = """
P1 (female, 21, undergraduate, engineering): "I plan everything in my head and then forget half of it.
I tried Notion but setting it up took longer than the assignment." Works part-time, studies at night.

P2 (male, 27, PhD candidate, biology): "My problem isn't tasks, it's the big picture - where is my thesis
in six months?" Uses Excel Gantt charts, meets supervisor every two weeks, feels anxious before meetings.

P3 (female, 24, master's, education): "I need reminders, but not annoying ones. And I like to see
progress, it motivates me." Uses phone calendar, studies in short bursts between family duties.

P4 (male, 20, undergraduate, business): "Group projects are the worst - nobody knows who does what."
Uses WhatsApp groups to coordinate, misses deadlines when teammates don't reply.

P5 (female, 30, PhD candidate, computer science): "I track my reading in Zotero, my code in GitHub,
my writing in Word. Nothing talks to each other." Wants one place to see research progress.

P6 (male, 22, undergraduate, medicine): "I study with spaced repetition. A planner that ignores
my exam dates is useless." Very structured, uses Anki daily.
"""

UPLOAD = False
if UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    RAW_DATA = "\n\n".join(content.decode("utf-8", errors="ignore") for content in uploaded.values())

print(f"Loaded {len(RAW_DATA):,} characters of research data.")

## Step 4 — Define the agents (each agent = a role prompt)

In [ ]:
ANALYST = """You are a senior UX researcher doing qualitative analysis.
Extract from the raw data: goals, pain points, behaviours, tools used, motivations, and context
of use. For every finding, cite the participant ID(s) and a short supporting quote.
Do not invent anything that is not in the data. Output clear Markdown sections."""

SEGMENTER = """You are a research strategist. Using the analysis, group participants into
2-4 distinct user segments based on meaningful differences in goals and behaviour
(not demographics alone). For each segment give: a working name, the participant IDs,
the defining characteristics, and why it is different from the other segments."""

PERSONA_WRITER = """You are an expert at writing research-based personas.
Write ONE persona for the given segment using this structure:
- Name, photo description, and a one-line tagline (a quote that captures them)
- Background (age range, role, context) - keep demographics realistic, not stereotyped
- Goals
- Frustrations / pain points
- Behaviours and current tools
- Needs from the product
- A short "day in the life" scenario
- Evidence: which participants and findings support this persona
Every claim must be traceable to the analysis. Output Markdown."""

CRITIC = """You are a critical reviewer of UX personas. Check the persona against the analysis:
1. Claims not supported by the evidence
2. Stereotypes or assumptions based on gender, age, or culture
3. Missing important findings from the segment
4. Vagueness - places where it is not actionable for designers
Give a numbered list of concrete fixes. If the persona is already strong, say "APPROVED"
and list only minor suggestions."""

## Step 5 — Orchestrate: run the agents as a pipeline

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import re

MAX_REVISIONS = 2   # how many critic -> writer rounds per persona

def build_persona(segment_name: str, segment_text: str, analysis: str) -> str:
    context = f"Research context:\n{RESEARCH_CONTEXT}\n\nAnalysis:\n{analysis}\n\nSegment:\n{segment_text}"
    persona = ask_agent(PERSONA_WRITER, context, effort="high")
    for round_no in range(1, MAX_REVISIONS + 1):
        review = ask_agent(CRITIC, f"{context}\n\nPersona to review:\n{persona}")
        print(f"  [{segment_name}] critic round {round_no}: "
              f"{'APPROVED' if 'APPROVED' in review else 'changes requested'}")
        if "APPROVED" in review:
            break
        persona = ask_agent(
            PERSONA_WRITER,
            f"{context}\n\nYour previous persona:\n{persona}\n\n"
            f"Reviewer feedback:\n{review}\n\nRewrite the full persona applying every fix.",
            effort="high",
        )
    return persona

# 1) Analyst
print("Analyst is reading the data...")
analysis = ask_agent(ANALYST, f"Research context:\n{RESEARCH_CONTEXT}\n\nRaw data:\n{RAW_DATA}", effort="high")

# 2) Segmenter
print("Segmenter is grouping participants...")
segments_md = ask_agent(
    SEGMENTER,
    f"Analysis:\n{analysis}\n\nStart each segment with a line '### Segment: <name>'.",
)
segments = [s.strip() for s in re.split(r"(?=### Segment:)", segments_md) if s.strip().startswith("### Segment:")]
print(f"Found {len(segments)} segments.")

# 3+4) Writer + Critic, one persona per segment, in parallel
def run(seg):
    name = seg.splitlines()[0].replace("### Segment:", "").strip()
    return name, build_persona(name, seg, analysis)

with ThreadPoolExecutor(max_workers=4) as pool:
    personas = list(pool.map(run, segments))

print("Done!")

## Step 6 — View the results

In [ ]:
from IPython.display import Markdown, display

display(Markdown("# Analysis\n" + analysis))
display(Markdown("# Segments\n" + segments_md))
for name, persona in personas:
    display(Markdown(f"---\n# Persona: {name}\n\n{persona}"))

## Step 7 — Save and download

In [ ]:
report = (
    "# Research Personas\n\n## Context\n" + RESEARCH_CONTEXT
    + "\n\n## Analysis\n" + analysis
    + "\n\n## Segments\n" + segments_md
    + "".join(f"\n\n---\n\n## Persona: {n}\n\n{p}" for n, p in personas)
)
with open("research_personas.md", "w", encoding="utf-8") as f:
    f.write(report)

try:
    from google.colab import files
    files.download("research_personas.md")
except ImportError:
    print("Saved to research_personas.md")